In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cantera as ct
import matplotlib.pyplot as plt

In [ ]:
# ── Configuration — must match h2o2_NN_toy_R22_discrete_targets_trainv2.py ───
YAML_FILE = 'chem_cti_toy_model_og.yaml'
IDX_R22   = 21

_gas_nom = ct.Solution(YAML_FILE)
NOMINAL_A_R22     = _gas_nom.reaction(IDX_R22).rate.low_rate.pre_exponential_factor
NOMINAL_B_R22     = _gas_nom.reaction(IDX_R22).rate.low_rate.temperature_exponent
NOMINAL_EA_R22_si = _gas_nom.reaction(IDX_R22).rate.low_rate.activation_energy
mol_units = ct.UnitSystem({
    'length': 'cm', 'mass': 'g', 'time': 's',
    'quantity': 'mol', 'pressure': 'dyn / cm^2', 'energy': 'erg',
    'temperature': 'K', 'current': 'A', 'activation-energy': 'cal / mol'})
NOMINAL_EA_R22_cal = mol_units.convert_activation_energy_to(
    f'{NOMINAL_EA_R22_si} J/kmol', 'cal / mol')
del _gas_nom

LN_F    = 10       # ln(f) uncertainty factor for A — same as training script
SIGMA_E = 2000.0   # cal/mol half-width for Ea   — same as training script

INPUT_DIM  = 2
HIDDEN_DIM = 16

T_INITIAL  = 1057
P_INITIAL  = 1.83 * ct.one_atm
INITIAL_X  = {'H2O2': 860e-6, 'H2O': 663e-6, 'O2': 332e-6,
               'AR':   1.0 - (860+663+332)*1e-6}
DT_MAX     = 1e-6
TIME_STEPS = 6000
T_SIM      = np.linspace(DT_MAX, DT_MAX * TIME_STEPS, TIME_STEPS)

EXP_CSV     = 'Hong et al_Burke.csv'
RESULT_PATH = 'gelu_discrete_targets.pt'

In [ ]:
# ── Experimental data ─────────────────────────────────────────────────────────
df_exp = pd.read_csv(EXP_CSV)
t_exp  = df_exp['time'].values           # seconds (already)
y_exp  = df_exp['x_h2o'].values * 1e-6  # ppm → mole fraction
print(f'Exp: {len(t_exp)} points, '
      f't=[{t_exp.min()*1e3:.3f}, {t_exp.max()*1e3:.3f}] ms, '
      f'H2O=[{y_exp.min()*1e6:.0f}, {y_exp.max()*1e6:.0f}] ppm')

In [ ]:
# ── Discrete-target NN architecture ──────────────────────────────────────────
# Mirrors SurrogateNN in the training script exactly.
class DiscreteNN(nn.Module):
    def __init__(self, hidden=HIDDEN_DIM, n_out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_DIM, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_out),
        )
    def forward(self, x):   # x: [B, 2]
        return self.net(x)  # [B, n_out]


def load_discrete_model(path):
    bundle       = torch.load(path, weights_only=False)
    target_times = bundle['target_times']            # seconds, shape [N_TARGETS]
    m = DiscreteNN(hidden=HIDDEN_DIM, n_out=len(target_times))
    m.load_state_dict(bundle['model_state'])
    m.eval()
    return m, target_times


def predict_discrete_ppm(model, x_vec, target_times):
    """Returns (ppm_array, t_ms_array) at the N_TARGETS discrete times."""
    x_t = torch.tensor(np.asarray(x_vec).reshape(1, -1), dtype=torch.float32)
    with torch.no_grad():
        raw = model(x_t).numpy().ravel()   # log-space outputs
    return np.exp(raw) * 1e6, target_times * 1e3


def perturb_reactions(gas, x_vec):
    new_A   = NOMINAL_A_R22 * np.exp(x_vec[0] * LN_F)
    new_Ea  = (NOMINAL_EA_R22_cal + x_vec[1] * SIGMA_E) * 4184.0
    rxn = gas.reaction(IDX_R22)
    rxn.rate.low_rate = ct.Arrhenius(new_A, NOMINAL_B_R22, new_Ea)
    gas.modify_reaction(IDX_R22, rxn)


def simulate_h2o(gas):
    h2o_idx = gas.species_index('H2O')
    gas.TPX = T_INITIAL, P_INITIAL, INITIAL_X
    reactor = ct.IdealGasReactor(gas, energy='on')
    net     = ct.ReactorNet([reactor])
    profile = np.empty(TIME_STEPS)
    for i in range(TIME_STEPS):
        net.advance(net.time + DT_MAX)
        profile[i] = reactor.thermo.X[h2o_idx]
    return profile

In [ ]:
# ── Load model and plot at x = [0, 0] ────────────────────────────────────────
disc_model, target_times = load_discrete_model(RESULT_PATH)
print(f'Loaded: {RESULT_PATH}')
print(f'N_TARGETS={len(target_times)}, target times (ms): {target_times*1e3}')

x_test_val = np.array([0.0, 0.0])

gas_nom  = ct.Solution(YAML_FILE)
y_nom_ct = simulate_h2o(gas_nom)

y_disc_ppm, t_disc_ms = predict_discrete_ppm(disc_model, x_test_val, target_times)
y_ct_at_targets = np.interp(target_times, T_SIM, y_nom_ct) * 1e6

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(T_SIM * 1e3, y_nom_ct * 1e6, 'r--', lw=1.5, label='Cantera nominal (x=0)')
ax.plot(t_exp * 1e3, y_exp * 1e6, 'o', mfc='none', mec='k',
        label='Exp — Hong et al.', zorder=5)
ax.scatter(t_disc_ms, y_disc_ppm, s=100, marker='D', color='steelblue',
           zorder=6, label=f'Discrete-target NN  (N={len(target_times)})')
ax.vlines(t_disc_ms, 0, y_disc_ppm, colors='steelblue', alpha=0.25, lw=1)

ax.set_xlabel('Time [ms]')
ax.set_ylabel(r'H$_2$O mole fraction [ppm]')
ax.set_xlim([0, 6])
ax.legend(loc='lower right', frameon=False)
ax.grid(True, ls='--', alpha=0.4)
ax.set_title(f'Discrete-target NN vs experiment  x={x_test_val}')
plt.tight_layout()
plt.show()

print(f'\n  {"t [ms]":>8}  {"Cantera [ppm]":>14}  {"NN [ppm]":>10}  {"rel err":>8}')
for t, y_ct, y_nn in zip(t_disc_ms, y_ct_at_targets, y_disc_ppm):
    err = abs(y_nn - y_ct) / (abs(y_ct) + 1e-30) * 100
    print(f'  {t:8.3f}  {y_ct:14.1f}  {y_nn:10.1f}  {err:7.2f}%')

In [ ]:
# ── Multi-point verification: 7 diagnostic test cases ────────────────────────
x_test_cases = {
    'nominal  [0, 0]':    [ 0.0,  0.0],
    'max A    [1, 0]':    [ 1.0,  0.0],
    'min A   [-1, 0]':    [-1.0,  0.0],
    'max Ea   [0, 1]':    [ 0.0,  1.0],
    'min Ea  [0, -1]':    [ 0.0, -1.0],
    'fast  [1, -1]':      [ 1.0, -1.0],
    'slow  [-1, 1]':      [-1.0,  1.0],
}

print('Running Cantera for 7 test points...')
ct_profiles = {}
for name, x_vec in x_test_cases.items():
    gas = ct.Solution(YAML_FILE)
    perturb_reactions(gas, np.array(x_vec))
    ct_profiles[name] = simulate_h2o(gas) * 1e6
    print(f'  {name:20s}  final={ct_profiles[name][-1]:.1f} ppm')

ncols = 2
nrows = (len(x_test_cases) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
axes_flat = axes.flatten()
t_ms_full = T_SIM * 1e3

for idx, (name, x_vec) in enumerate(x_test_cases.items()):
    ax = axes_flat[idx]
    y_ct = ct_profiles[name]

    y_nn_ppm, t_nn_ms = predict_discrete_ppm(disc_model, np.array(x_vec), target_times)
    y_ct_at_t = np.interp(target_times, T_SIM, y_ct / 1e6) * 1e6
    rel_err = np.abs(y_nn_ppm - y_ct_at_t) / (np.abs(y_ct_at_t) + 1e-30) * 100

    ax.plot(t_ms_full, y_ct, color='#333333', lw=2, label='Cantera')
    ax.scatter(t_nn_ms, y_nn_ppm, s=70, marker='D', color='steelblue', zorder=5,
               label=f'NN  (εmean={rel_err.mean():.1f}%)')
    ax.vlines(t_nn_ms, 0, y_nn_ppm, colors='steelblue', alpha=0.25, lw=1)

    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Time [ms]'); ax.set_ylabel(r'H$_2$O [ppm]')
    ax.set_xlim([0, 6]); ax.grid(True, ls='--', alpha=0.3)
    ax.legend(fontsize=8, frameon=False, loc='lower right')

for idx in range(len(x_test_cases), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Discrete-target NN vs Cantera — 7 test points', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('discrete_nn_7points.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: discrete_nn_7points.png')